[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/14_kv_cache_solution.ipynb)

# 🔴 Solution: KV Cache Attention

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `14_kv_cache.ipynb` first.

---
Implement multi-head attention with a **KV cache** for incremental decoding.

### Signature
```python
class KVCacheAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs): ...
    def __call__(self, x, cache=None): ...   # -> (out, new_cache)
```

- `x`: `(B, S_new, d_model)` — only the **new** tokens
- `cache`: `None`, or `(k, v)` each `(B, H, S_past, d_k)`
- returns `out` of shape `(B, S_new, d_model)` and the updated `(k, v)`

### Requirements
- Four `nnx.Linear(d_model, d_model)` layers: `W_q`, `W_k`, `W_v`, `W_o`
- Concatenate the cached `k`/`v` along the **sequence** axis
- Scale by $1/\sqrt{d_k}$
- Causal mask when `S_new > 1`, offset by `S_past`

`nnx.Linear` is an allowed building block — the exercise is the cache, not the
projection.

### Why the cache exists
Generating token $n$ re-attends over all $n-1$ previous tokens. Without a cache
you recompute every past key and value at every step, making generation
$O(n^2)$ per token and $O(n^3)$ overall. With a cache each step projects only
the new token and appends, so generation is $O(n)$ per step.

### The mask offset — the part people get wrong
During **decode** (`S_new == 1`) there is nothing to mask: the single query is
the newest position and may see everything before it.

During **prefill** (`S_new > 1`) the score matrix is `(S_new, S_total)` and is
**not square**. Query $i$ sits at absolute position $S_{past} + i$, so it may
attend to key $j$ only when $j \le S_{past} + i$. That is a triangular mask
shifted right by `S_past` — using an unshifted `triu` silently blocks the
cached history and is the classic bug here.

### The memory arithmetic
Cache size is
$2 \times L \times B \times H_{kv} \times S \times d_k \times \text{bytes}$
— the leading 2 is K and V, and $L$ is the layer count, which is easy to drop
and worth a factor of 80.

For Llama-3-70B in bf16 ($L=80$, $H_{kv}=8$, $d_k=128$, 2 bytes) that is
$2 \times 80 \times 8 \times 128 \times 2 = 327{,}680$ bytes = **320 KiB per
token** — roughly 10 GiB at 32k context, per sequence. At batch 32 the
cache dwarfs the weights, which is why GQA, int8 KV and PagedAttention all
exist.

### A JAX note on this design
The cache here is a **value** passed in and returned, not mutable state hidden
in the module — which is exactly how JAX prefers it. The cost is that `k` grows
by one each step, so a jitted decode loop retraces on every new shape.
Production code preallocates a fixed-size buffer and writes into it with
`jax.lax.dynamic_update_slice`, trading wasted flops on empty slots for a
single compilation. This task keeps the concat version to match the original;
the preallocated variant is the natural follow-up question.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class KVCacheAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs):
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_k = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_v = nnx.Linear(d_model, d_model, rngs=rngs)
        self.W_o = nnx.Linear(d_model, d_model, rngs=rngs)

    def _heads(self, t, B, S):
        # (B, S, d_model) -> (B, H, S, d_k)
        return t.reshape(B, S, self.num_heads, self.d_k).transpose(0, 2, 1, 3)

    def __call__(self, x, cache=None):
        B, S_new, _ = x.shape

        q = self._heads(self.W_q(x), B, S_new)
        k = self._heads(self.W_k(x), B, S_new)
        v = self._heads(self.W_v(x), B, S_new)

        # The cache is a VALUE we extend, not mutable state.
        if cache is not None:
            k = jnp.concatenate([cache[0], k], axis=2)
            v = jnp.concatenate([cache[1], v], axis=2)

        new_cache = (k, v)
        S_total = k.shape[2]

        # == q @ jnp.swapaxes(k, -1, -2), written as a contraction.
        scores = jnp.einsum("bhtd,bhsd->bhts", q, k) / jnp.sqrt(
            jnp.asarray(self.d_k, x.dtype)
        )

        if S_new > 1:
            # Query i is at absolute position S_past + i, so the triangle is
            # shifted right by S_past. Unshifted triu would hide the cache.
            S_past = S_total - S_new
            blocked = jnp.triu(
                jnp.ones((S_new, S_total), dtype=bool), k=S_past + 1
            )
            scores = jnp.where(blocked, -jnp.inf, scores)

        weights = jax.nn.softmax(scores, axis=-1)
        attn = jnp.einsum("bhts,bhsd->bhtd", weights, v)   # == weights @ v

        merged = attn.transpose(0, 2, 1, 3).reshape(B, S_new, -1)
        return self.W_o(merged), new_cache

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

m = KVCacheAttention(32, 4, rngs=nnx.Rngs(params=0))

# Prefill 5 tokens, then decode 3 more one at a time.
x = jax.random.normal(jax.random.key(1), (1, 5, 32))
out, cache = m(x)
print("prefill:", out.shape, "cache k:", cache[0].shape)

for step in range(3):
    tok = jax.random.normal(jax.random.key(10 + step), (1, 1, 32))
    out, cache = m(tok, cache)
    print(f"  step {step}: out {out.shape}  cache grew to {cache[0].shape[2]}")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("kv_cache")